# 06 – Statistical tests

**Goal:** Test the first hypotheses with classical tests, each with p-value **and** effect size (α = 0.05).
**Input:** PostgreSQL tables `pairs`, `races`, `circuits` · **Output:** summary table of all tests
**Assessment items:** B4 (Chi² test, ANOVA, correlation analysis with p-values), basis for P7 and K6

| # | Test | Question | H0 |
|---|---|---|---|
| 1 | Chi² test of independence | Does the overtake rate depend on the era? (pair-lap level) | equal rate in E1, E2, E3 |
| 2 | One-way ANOVA / Kruskal-Wallis | Do the per-race overtake rates differ between eras? (race level) | equal means / distributions |
| 3 | Correlation (Pearson, Spearman) | Is the gap at lap start related to the overtake rate? | ρ = 0 |
| 4 | Spearman (race level, E2 only) | Does the overtake rate fall from 2022 to 2025? (sub-question 2) | ρ = 0 |
| 5 | Welch t-test / Mann-Whitney U | Street vs. permanent circuits | equal mean rate per race |
| 6 | Spearman (circuit level) | Is a longer straight related to more overtakes? | ρ = 0 |

**Why two levels?** Pair-laps from the same race are not independent (same weather, track, safety-car phases). Tests on 70,000 pair-laps therefore produce very small p-values even for small differences. Each pair-lap result is complemented by a test with **races as independent units**, and effect sizes are reported for every test.

In [1]:
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir())
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
from scipy import stats

pd.set_option("display.width", 200)

In [2]:
from statsmodels.stats.multicomp import pairwise_tukeyhsd

from src.db import get_engine

ALPHA = 0.05
engine = get_engine()
results = []  # summary rows, filled by every test


def record(test, question, statistic, p, effect, effect_name, n):
    """Store one test result for the summary table."""
    results.append({"test": test, "question": question, "statistic": round(statistic, 3),
                    "p_value": p, "effect_size": round(effect, 3), "effect_measure": effect_name,
                    "n": n, "reject_H0": p < ALPHA})

In [3]:
pairs = pd.read_sql("""
    SELECT p.overtake, p.gap_s, r.race_id, r.season, r.era, c.circuit_id, c.circuit_type,
           c.longest_straight_m
    FROM pairs p
    JOIN races r    USING (race_id)
    JOIN circuits c USING (circuit_id)
""", engine)

# race level: one row per race, aggregated in SQL
races = pd.read_sql("""
    SELECT r.race_id, r.season, r.era, c.circuit_type,
           COUNT(*)                AS pair_laps,
           SUM(p.overtake)         AS overtakes,
           AVG(p.overtake)::float  AS rate
    FROM pairs p
    JOIN races r    USING (race_id)
    JOIN circuits c USING (circuit_id)
    GROUP BY r.race_id, r.season, r.era, c.circuit_type
""", engine)
print(len(pairs), "pair-laps,", len(races), "races")

70196 pair-laps, 192 races


## 1. Chi² test: overtake × era (pair-lap level)

In [4]:
table = pd.crosstab(pairs["era"], pairs["overtake"])
chi2, p, dof, expected = stats.chi2_contingency(table)
n = table.values.sum()
cramers_v = np.sqrt(chi2 / (n * (min(table.shape) - 1)))
print(table)
print(f"\nchi2 = {chi2:.2f}, df = {dof}, p = {p:.2e}, Cramér's V = {cramers_v:.3f}")
print("smallest expected frequency:", round(expected.min(), 1), "(assumption: all >= 5)")
record("Chi² test of independence", "overtake × era", chi2, p, cramers_v, "Cramér's V", n)

overtake      0     1
era                  
E1        25469  2088
E2        33340  2743
E3         5925   631

chi2 = 34.27, df = 2, p = 3.62e-08, Cramér's V = 0.022
smallest expected frequency: 510.1 (assumption: all >= 5)


In [5]:
# Post-hoc: pairwise 2x2 Chi² tests with Bonferroni correction
rows = []
for a, b in [("E1", "E2"), ("E2", "E3"), ("E1", "E3")]:
    sub = table.loc[[a, b]]
    c2, p_pair, _, _ = stats.chi2_contingency(sub)
    rate_a, rate_b = (sub[1] / sub.sum(axis=1)).values
    rows.append({"pair": f"{a} vs {b}", "rate_a_pct": round(100 * rate_a, 2),
                 "rate_b_pct": round(100 * rate_b, 2), "chi2": round(c2, 2),
                 "p_bonferroni": min(1.0, 3 * p_pair)})
pd.DataFrame(rows)

,pair,rate_a_pct,rate_b_pct,chi2,p_bonferroni
0,E1 vs E2,7.58,7.60,0.01,1.000000e+00
1,E2 vs E3,7.60,9.62,30.88,8.243326e-08
2,E1 vs E3,7.58,9.62,30.00,1.298297e-07


**Reading:** The overall test rejects H0, but the effect is very small (Cramér's V well below 0.1). The post-hoc tests show where the difference comes from: E1 and E2 do not differ, E3 (2026) lies above both. Because pair-laps within a race are dependent, the race-level test below is the stricter check.

## 2. ANOVA / Kruskal-Wallis: overtake rate per race × era (race level)

In [6]:
groups = {era: g["rate"].values for era, g in races.groupby("era")}
summary = races.groupby("era")["rate"].agg(["count", "mean", "median", "std"]).mul(
    [1, 100, 100, 100]).round(2)
summary.columns = ["races", "mean_rate_pct", "median_rate_pct", "sd_pct"]
print(summary, "\n")

# Assumptions: normality per group (Shapiro-Wilk) and equal variances (Levene)
for era, values in groups.items():
    w, p_norm = stats.shapiro(values)
    print(f"Shapiro-Wilk {era}: W = {w:.3f}, p = {p_norm:.3f}")
lev, p_lev = stats.levene(*groups.values())
print(f"Levene: W = {lev:.3f}, p = {p_lev:.3f}")

     races  mean_rate_pct  median_rate_pct  sd_pct
era                                               
E1      81           8.21             8.25    4.27
E2      92           8.03             7.70    3.83
E3      19           9.85            10.70    3.95 

Shapiro-Wilk E1: W = 0.976, p = 0.130
Shapiro-Wilk E2: W = 0.977, p = 0.108
Shapiro-Wilk E3: W = 0.987, p = 0.994
Levene: W = 0.267, p = 0.766


In [7]:
f, p_anova = stats.f_oneway(*groups.values())
grand = races["rate"].mean()
ss_between = sum(len(v) * (v.mean() - grand) ** 2 for v in groups.values())
ss_total = ((races["rate"] - grand) ** 2).sum()
eta_sq = ss_between / ss_total

h, p_kw = stats.kruskal(*groups.values())
k, n_r = len(groups), len(races)
epsilon_sq = (h - k + 1) / (n_r - k)

print(f"ANOVA:          F = {f:.2f}, p = {p_anova:.4f}, eta² = {eta_sq:.3f}")
print(f"Kruskal-Wallis: H = {h:.2f}, p = {p_kw:.4f}, epsilon² = {epsilon_sq:.3f}")
record("One-way ANOVA", "rate per race × era", f, p_anova, eta_sq, "eta²", n_r)
record("Kruskal-Wallis", "rate per race × era", h, p_kw, epsilon_sq, "epsilon²", n_r)

ANOVA:          F = 1.63, p = 0.1983, eta² = 0.017
Kruskal-Wallis: H = 3.87, p = 0.1444, epsilon² = 0.010


In [8]:
tukey = pairwise_tukeyhsd(races["rate"], races["era"], alpha=ALPHA)
pd.DataFrame(tukey.summary().data[1:], columns=tukey.summary().data[0])

,group1,group2,meandiff,p-adj,lower,upper,reject
0,E1,E2,-0.0018,0.9556,-0.0163,0.0127,False
1,E1,E3,0.0164,0.2482,-0.0078,0.0407,False
2,E2,E3,0.0182,0.1750,-0.0058,0.0422,False


**Reading:** At race level the eras E1 and E2 are practically identical. E3 has the highest mean rate; whether the difference is significant with only 19 races in 2026 is shown by Tukey's test and the Kruskal-Wallis test (robust if the normality assumption is violated). The effect size (eta²) states how much of the variation between races the era explains – the rest comes from circuits, weather and race events, which the models will separate.

## 3. Correlation: gap at lap start and overtaking

In [9]:
# (a) pair-lap level: rank correlation between gap and the binary target
rho, p_rho = stats.spearmanr(pairs["gap_s"], pairs["overtake"])
print(f"pair level   Spearman rho = {rho:.3f}, p = {p_rho:.2e}, n = {len(pairs)}")
record("Spearman correlation", "gap_s × overtake (pair-laps)", rho, p_rho, rho, "rho", len(pairs))

# (b) classes of 0.1 s: gap class midpoint vs. overtake rate
cls = pd.cut(pairs["gap_s"], np.arange(0, 2.0001, 0.1), include_lowest=True)
binned = pairs.groupby(cls, observed=True)["overtake"].mean().reset_index()
binned["mid"] = [c.mid for c in binned["gap_s"]]
r_p, p_p = stats.pearsonr(binned["mid"], binned["overtake"])
r_s, p_s = stats.spearmanr(binned["mid"], binned["overtake"])
print(f"gap classes  Pearson r = {r_p:.3f} (p = {p_p:.2e}), Spearman rho = {r_s:.3f} "
      f"(p = {p_s:.2e}), n = {len(binned)} classes")
record("Pearson correlation", "gap class × overtake rate", r_p, p_p, r_p, "r", len(binned))
record("Spearman correlation", "gap class × overtake rate", r_s, p_s, r_s, "rho", len(binned))

pair level   Spearman rho = -0.316, p = 0.00e+00, n = 70196


gap classes  Pearson r = -0.826 (p = 7.30e-06), Spearman rho = -0.995 (p = 7.23e-20), n = 20 classes


**Reading:** The relation is strongly negative and monotonic (Spearman close to −1 across the gap classes). It is not linear – the rate falls steeply below one second and flattens out above – so Spearman describes it better than Pearson, and the logistic regression in notebook 07 is the appropriate model.

## 4. Trend within E2: did the ground-effect advantage fade? (race level)

In [10]:
e2 = races.query("era == 'E2'")
rho_e2, p_e2 = stats.spearmanr(e2["season"], e2["rate"])
tau_e2, p_tau = stats.kendalltau(e2["season"], e2["rate"])
print(e2.groupby("season")["rate"].agg(["count", "mean"]).assign(mean=lambda d: (100 * d["mean"]).round(2)))
print(f"\nSpearman rho = {rho_e2:.3f}, p = {p_e2:.4f}; Kendall tau = {tau_e2:.3f}, p = {p_tau:.4f}")
record("Spearman correlation", "season × rate per race (E2)", rho_e2, p_e2, rho_e2, "rho", len(e2))

        count  mean
season             
2022       22  9.11
2023       22  8.82
2024       24  7.82
2025       24  6.53

Spearman rho = -0.209, p = 0.0457; Kendall tau = -0.162, p = 0.0395


## 5. Street vs. permanent circuits (race level)

In [11]:
street = races.loc[races["circuit_type"] == "street", "rate"]
race_c = races.loc[races["circuit_type"] == "race", "rate"]
t, p_t = stats.ttest_ind(race_c, street, equal_var=False)
u, p_u = stats.mannwhitneyu(race_c, street, alternative="two-sided")
pooled_sd = np.sqrt((race_c.var(ddof=1) + street.var(ddof=1)) / 2)
cohens_d = (race_c.mean() - street.mean()) / pooled_sd
# rank-biserial correlation: > 0 means race circuits tend to have higher rates
rank_biserial = 2 * u / (len(race_c) * len(street)) - 1
print(f"mean rate: race circuits {100 * race_c.mean():.2f} % (n = {len(race_c)}), "
      f"street circuits {100 * street.mean():.2f} % (n = {len(street)})")
print(f"Welch t = {t:.2f}, p = {p_t:.2e}, Cohen's d = {cohens_d:.2f}")
print(f"Mann-Whitney U = {u:.0f}, p = {p_u:.2e}, rank-biserial r = {rank_biserial:.2f}")
record("Welch t-test", "rate per race: race vs. street circuit", t, p_t, cohens_d, "Cohen's d", len(races))
record("Mann-Whitney U", "rate per race: race vs. street circuit", u, p_u, rank_biserial,
       "rank-biserial r", len(races))

mean rate: race circuits 9.06 % (n = 141), street circuits 6.13 % (n = 51)
Welch t = 4.53, p = 1.91e-05, Cohen's d = 0.75
Mann-Whitney U = 5092, p = 1.09e-05, rank-biserial r = 0.42


## 6. Circuit level: longest straight and overtake rate

In [12]:
circ = pairs.groupby("circuit_id").agg(rate=("overtake", "mean"),
                                       straight=("longest_straight_m", "first"),
                                       races=("race_id", "nunique")).query("races >= 3")
rho_c, p_c = stats.spearmanr(circ["straight"], circ["rate"])
print(f"Spearman rho = {rho_c:.3f}, p = {p_c:.4f}, n = {len(circ)} circuits")
record("Spearman correlation", "longest straight × rate (circuits)", rho_c, p_c, rho_c, "rho", len(circ))

Spearman rho = 0.487, p = 0.0116, n = 26 circuits


## 7. Summary

In [13]:
summary = pd.DataFrame(results)
summary["p_value"] = summary["p_value"].map(
    lambda p: "< 1e-300" if p == 0 else (f"{p:.2e}" if p < 0.001 else f"{p:.4f}"))
summary

,test,question,statistic,p_value,effect_size,effect_measure,n,reject_H0
0,Chi² test of independence,overtake × era,34.270,3.62e-08,0.022,Cramér's V,70196,True
1,One-way ANOVA,rate per race × era,1.632,0.1983,0.017,eta²,192,False
2,Kruskal-Wallis,rate per race × era,3.871,0.1444,0.010,epsilon²,192,False
3,Spearman correlation,gap_s × overtake (pair-laps),-0.316,< 1e-300,-0.316,rho,70196,True
4,Pearson correlation,gap class × overtake rate,-0.826,7.30e-06,-0.826,r,20,True
5,Spearman correlation,gap class × overtake rate,-0.995,7.23e-20,-0.995,rho,20,True
6,Spearman correlation,season × rate per race (E2),-0.209,0.0457,-0.209,rho,92,True
7,Welch t-test,rate per race: race vs. street circuit,4.533,1.91e-05,0.751,Cohen's d,192,True
8,Mann-Whitney U,rate per race: race vs. street circuit,5092.000,1.09e-05,0.416,rank-biserial r,192,True
9,Spearman correlation,longest straight × rate (circuits),0.487,0.0116,0.487,rho,26,True


## 8. Interpretation (status: 2026 up to the latest race)

1. **Gap (H1):** the strongest and most robust relation – the closer the car behind, the more likely the overtake, steeply below one second. This supports H1 and motivates a non-linear (logistic) model.
2. **Era (H2, H3):** at pair-lap level the overtake rate differs significantly between eras, but the effect is negligible (Cramér's V ≈ 0.02) and comes only from 2026. At race level – the stricter test because pair-laps of one race are dependent – the eras do **not** differ significantly (ANOVA and Kruskal-Wallis, p > 0.05). The ground-effect cars of 2022 did *not* produce more overtakes per close-following situation than the cars of 2018–2021. Whether 2026 really differs can only be judged with the full season and with the models that control for gap, tyres and circuit.
3. **Trend within E2:** the per-race overtake rate fell significantly from 2022 to 2025 (Spearman, p < 0.05) – consistent with the idea that the effect of the 2022 rules faded as teams developed the cars.
4. **Circuit (H4):** permanent circuits have a clearly higher overtake rate than street circuits (medium-to-large effect, Cohen's d ≈ 0.75), and circuits with longer straights have more overtakes (ρ ≈ 0.5). The circuit explains more than the era – first support for H4.

These are bivariate results. The multivariate models (notebook 07) test whether they hold when all factors are considered together.